<a href="https://colab.research.google.com/github/vertgram/Pius_Olasupo/blob/main/lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4 — Linear Models and Regularization
**CAMLDS Machine Learning Class · ML 2/8**

---

**Dataset:** Olist Brazilian E-commerce (Kaggle)
**Session time:** 2 hours
**Input:** `orders_full`, `delivered`, and the `is_late` target, all built in Sessions 1 through 3
**New this session:** `estimated_days` (from Session 3, carried forward) becomes a modeling feature; `olist_order_reviews_dataset.csv` returns in the Challenge Block

**By the end of this lab you will be able to:**
1. Fit a linear regression with scikit-learn and interpret what each coefficient means.
2. Fit a logistic regression for a binary target and interpret its coefficients.
3. Explain what L1 (Lasso) and L2 (Ridge) regularization penalize, and see the difference on a real feature set.
4. Describe what gradient descent does at an intuitive level, and watch it converge on real data.

---

> **Reference:** 3Blue1Brown, "Essence of Calculus" playlist (gradient intuition); StatQuest, "Regularization" videos, Ridge, Lasso, ElasticNet (Josh Starmer, YouTube)

## Block 0 — Environment Setup

Run this cell first. Today adds four scikit-learn tools to the ones you already know: `LinearRegression`, `LogisticRegression`, `Ridge`, and `Lasso`, plus `StandardScaler` for putting features on the same scale before regularizing them.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['figure.dpi'] = 100

pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'pandas {pd.__version__}  |  scikit-learn {sklearn.__version__}  |  matplotlib {plt.matplotlib.__version__}')
print('Setup complete.')

pandas 3.0.3  |  scikit-learn 1.9.0  |  matplotlib 3.10.9
Setup complete.


---

## Block 1 — Rebuild orders_full, is_late, and estimated_days

Session 4 starts from the table you built across the first three sessions, plus one column from Session 3 you have already seen: `estimated_days`, the promised delivery window from checkout. We rebuild all of it in one hidden block below so today's lab can get straight to fitting models.

**[instructor] Before class:** upload the five CSVs to Drive and replace the gdown IDs below. The reviews table is not read into a DataFrame until the Challenge Block, same as Session 3.

**A note on splits today.** We fit several models this session, so we use one 80/20 train and validation split per model rather than redoing the full train, validation, and test split from Session 3 each time. That three-way discipline still applies once you commit to a single model to ship, which is exactly what Session 10's capstone will ask you to do.

In [ ]:
# ── Download from Drive (replace IDs before class; skips files already on disk) ──
import subprocess, os

FILES = {
    'olist_orders_dataset.csv'                        : 'GDOWN_ID_ORDERS',
    'olist_order_items_dataset.csv'                   : 'GDOWN_ID_ITEMS',
    'olist_customers_dataset.csv'                     : 'GDOWN_ID_CUSTOMERS',
    'olist_order_payments_dataset.csv'                : 'GDOWN_ID_PAYMENTS',
    'olist_order_reviews_dataset.csv'                 : 'GDOWN_ID_REVIEWS',
}

for fname, gid in FILES.items():
    if os.path.exists(fname):
        print(f'{fname} already present, skipping download')
        continue
    url = f'https://drive.google.com/uc?id={gid}'
    subprocess.run(['gdown', url, '-O', fname], check=True)
    print(f'Downloaded {fname}')

olist_orders_dataset.csv already present, skipping download
olist_order_items_dataset.csv already present, skipping download
olist_customers_dataset.csv already present, skipping download
olist_order_payments_dataset.csv already present, skipping download
olist_order_reviews_dataset.csv already present, skipping download


In [ ]:
# Rebuild the Session 1-3 tables
orders    = pd.read_csv('olist_orders_dataset.csv')
items     = pd.read_csv('olist_order_items_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
payments  = pd.read_csv('olist_order_payments_dataset.csv')

# 1. Cast timestamps (Session 1, Block 4)
timestamp_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

orders_clean = orders.dropna(subset=['order_purchase_timestamp']).copy()

# 2. Per-order item summary (Session 1, Block 7)
items['freight_value'] = items['freight_value'].fillna(0)
items['total_cost']    = items['price'] + items['freight_value']
order_summary = (
    items.groupby('order_id')
    .agg(item_count=('order_item_id', 'count'),
         total_price=('price', 'sum'),
         total_freight=('freight_value', 'sum'),
         total_cost=('total_cost', 'sum'))
    .reset_index()
)

# 3. Per-order payment summary (Session 1, Challenge Block)
payments_per_order = (
    payments.groupby('order_id')
    .agg(dominant_payment_type=('payment_type', lambda x: x.value_counts().index[0]),
         total_payment_value=('payment_value', 'sum'),
         max_installments=('payment_installments', 'max'))
    .reset_index()
)

# 4. Merge into one flat table (Session 1, Block 9)
orders_full = (
    orders_clean
    .merge(order_summary, on='order_id', how='left')
    .merge(customers[['customer_id', 'customer_state', 'customer_city']],
           on='customer_id', how='left')
    .merge(payments_per_order, on='order_id', how='left')
)

# 5. Delivery time in days (Session 2, Block 1)
orders_full['delivery_days'] = (
    (orders_full['order_delivered_customer_date'] - orders_full['order_purchase_timestamp'])
    .dt.days
)

print(f'orders_full: {orders_full.shape[0]:,} rows x {orders_full.shape[1]} columns')

# 6. Rebuild is_late (Session 2, Challenge Block reveal). is_late only makes sense
#    for orders that were actually delivered, so this is our working table.
delivered = orders_full[orders_full['order_delivered_customer_date'].notna()].copy()
delivered['is_late'] = (
    delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']
)

late_rate = delivered['is_late'].mean()
print(f'delivered orders: {len(delivered):,}')
print(f'late orders: {delivered["is_late"].sum():,} ({late_rate*100:.2f}% of delivered orders)')

# 7. estimated_days (Session 3, Block 5): the promised delivery window, known at
#    order time. Today's features use it, so we carry it forward here.
delivered['estimated_days'] = (
    (delivered['order_estimated_delivery_date'] - delivered['order_purchase_timestamp'])
    .dt.days
)
print(f'estimated_days: mean {delivered["estimated_days"].mean():.1f} days, '
      f'median {delivered["estimated_days"].median():.0f} days')

orders_full: 99,441 rows x 18 columns
delivered orders: 96,476
late orders: 7,827 (8.11% of delivered orders)
estimated_days: mean 23.4 days, median 23 days


`delivered` is our working table again today. `estimated_days` and `delivery_days` are both measured in days: one is the promise made at checkout, the other is what actually happened. Today's first two models both use `estimated_days` as a feature, since it is known at order time and Session 3 already showed it carries real signal.

---

## Block 2 — Linear Regression: Predicting delivery_days

Linear regression learns a straight-line relationship between a set of input columns and a numeric target. It picks one coefficient per feature and one intercept, and adds them up to produce a prediction. The coefficient on a feature tells you how much the prediction changes when that feature goes up by one unit, holding the other features fixed.

Today's target is `delivery_days`, the actual number of days an order took to arrive. Our features are four columns known at order time: `item_count`, `total_price`, `total_freight`, and `estimated_days`.

In [ ]:
feature_cols = ['item_count', 'total_price', 'total_freight', 'estimated_days']
target_col = 'delivery_days'

model_data = delivered.dropna(subset=feature_cols + [target_col]).copy()
train, val = train_test_split(model_data, test_size=0.20, random_state=42)

lin_reg = LinearRegression().fit(train[feature_cols], train[target_col])
val_pred = lin_reg.predict(val[feature_cols])

print('Coefficients:')
for feature, coef in zip(feature_cols, lin_reg.coef_):
    print(f'  {feature:15}: {coef:8.4f}')
print(f'  intercept:       {lin_reg.intercept_:8.4f}')

mean_baseline = np.full(len(val), train[target_col].mean())
print(f'\nValidation MAE, linear regression: {mean_absolute_error(val[target_col], val_pred):.2f} days')
print(f'Validation MAE, mean predictor:    {mean_absolute_error(val[target_col], mean_baseline):.2f} days')
print(f'Validation R-squared:              {r2_score(val[target_col], val_pred):.3f}')

Coefficients:
  item_count     :  -1.3052
  total_price    :  -0.0005
  total_freight  :   0.0515
  estimated_days :   0.3907
  intercept:         3.3310

Validation MAE, linear regression: 5.62 days
Validation MAE, mean predictor:    6.47 days
Validation R-squared:              0.140


In [ ]:
feature

(96476, 20)

The linear regression beats the mean predictor baseline, 5.62 days of mean absolute error against 6.47 days. That is a real improvement, but the R-squared of about 0.14 tells you these four features only explain a small share of why delivery time varies. Most of what determines how long a package takes, traffic, warehouse handling, the carrier's route that week, is not in this feature set at all. This matches what Session 3 found about the same relationship: real delivery data is noisy, and no small feature set captures all of it.

Read the coefficients one at a time, holding everything else fixed:
- `estimated_days`: +0.39. Each extra day in the promised delivery window is associated with about 0.39 extra actual days. Orders with a longer promised window tend to genuinely take longer, most likely because a long estimate reflects a farther or slower route in the first place.
- `total_freight`: +0.05. Slightly higher freight cost, which often means a heavier or farther shipment, is associated with a small increase in delivery time.
- `item_count`: -1.31. Orders with more items arrive slightly faster on average. This is counterintuitive at first, but multi-item orders are often shipped from a single warehouse together, while a single-item order might come from a farther seller.
- `total_price`: -0.0005. Close to zero. Price alone carries almost no information about delivery time once the other three features are accounted for.

In [ ]:
# Exercise 2.1
# Refit the linear regression WITHOUT estimated_days (keep item_count, total_price,
# and total_freight only). How much does validation MAE get worse? What does that
# tell you about how much of the earlier model's signal came from estimated_days alone?

# TODO: your code here


---

## Block 3 — Logistic Regression: Predicting is_late

Logistic regression is built for a binary target, one that is only ever one of two values. Instead of predicting a number directly, it predicts a probability between 0 and 1, then applies a threshold, 0.5 by default, to turn that probability into a yes or no answer.

Today's target is `is_late`, back from Session 2 and 3. We reuse the same four features as Block 2. Logistic regression coefficients are easiest to compare when every feature is on the same scale, so we standardize them first with `StandardScaler`: it subtracts each column's mean and divides by its standard deviation, so every feature ends up centered at 0 with a standard deviation of 1.

In [ ]:
feature_cols = ['item_count', 'total_price', 'total_freight', 'estimated_days']
target_col = 'is_late'

model_data = delivered.dropna(subset=feature_cols + [target_col]).copy()
train, val = train_test_split(
    model_data, test_size=0.20, stratify=model_data[target_col], random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_val = scaler.transform(val[feature_cols])

log_reg = LogisticRegression().fit(X_train, train[target_col])
val_pred = log_reg.predict(X_val)

print('Coefficients (on standardized features):')
for feature, coef in zip(feature_cols, log_reg.coef_[0]):
    print(f'  {feature:15}: {coef:8.4f}')

val_acc = accuracy_score(val[target_col], val_pred)
majority_baseline = 1 - val[target_col].mean()
print(f'\nValidation accuracy, logistic regression: {val_acc*100:.2f}%')
print(f'Majority class baseline:                  {majority_baseline*100:.2f}%')
print(f'Late orders in validation set caught at the default 0.5 threshold: '
      f'{(val_pred & val[target_col]).sum()} of {val[target_col].sum()}')

This result needs a careful read. Accuracy comes out at 91.86%, just under the 91.89% majority baseline that always predicts "not late." At the default 0.5 threshold, the model catches 0 of the 1,565 late orders in the validation set. It is not doing better than Block 3's naive baseline, at least not on accuracy.

That does not mean the model learned nothing. The coefficients tell a coherent story: `estimated_days` is negative (-0.30), meaning a longer promised window is associated with a lower predicted probability of lateness, since a generous estimate is easier to hit. `total_freight` is positive (+0.19), so heavier or farther shipments look slightly riskier. These signs make sense, and they match what Block 2's linear regression found in the opposite direction.

The real problem is the 8.11% late rate you first saw in Session 3. With such an imbalanced target, a model can assign every order a low probability of being late, mostly correctly, and still never cross the 0.5 threshold for the rare late cases. Accuracy alone cannot tell you whether a model is actually useful here. Session 6 covers metrics built exactly for this problem, precision, recall, and ROC-AUC, that judge a model on how well it catches the minority class, not just on how often it is right overall.

In [ ]:
# Exercise 3.1
# Get the predicted probabilities with log_reg.predict_proba(X_val)[:, 1], then try
# thresholds of 0.2 and 0.1 instead of the default 0.5. For each threshold, report
# accuracy and how many of the 1,565 actually-late orders in val get caught. What
# trade-off do you see between catching more late orders and overall accuracy?

# TODO: your code here


---

## Block 4 — Regularization: Ridge and Lasso

A plain linear regression finds whatever coefficients best fit the training data, with no penalty for how large those coefficients get. That becomes a problem when features overlap in what they measure. **Regularization** adds a penalty for large coefficients directly into what the model optimizes, which trades a small amount of fit on the training data for a model that is more stable and easier to trust.

Two standard penalties cover most cases:
- **L2, used by Ridge.** Penalizes the sum of squared coefficients. This shrinks every coefficient toward zero, smoothly, but rarely sets one to exactly zero.
- **L1, used by Lasso.** Penalizes the sum of absolute coefficients. This can drive some coefficients to exactly zero, which removes those features from the model entirely. Lasso is a way to do feature selection while fitting the model, not as a separate step.

We expand the feature set to include three columns that overlap heavily with each other: `total_cost` (already defined as `total_price` plus `total_freight`), `total_payment_value`, and `max_installments`. Because regularization penalizes raw coefficient size, we standardize every feature first, the same way Block 3 did, so that a feature measured in reais does not get penalized differently than one measured in days.

In [ ]:
feature_cols = [
    'item_count', 'total_price', 'total_freight', 'total_cost',
    'total_payment_value', 'max_installments', 'estimated_days',
]
target_col = 'delivery_days'

model_data = delivered.dropna(subset=feature_cols + [target_col]).copy()
train, val = train_test_split(model_data, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_val = scaler.transform(val[feature_cols])
y_train, y_val = train[target_col], val[target_col]

lin_reg = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
lasso = Lasso(alpha=0.1).fit(X_train, y_train)

print(f"{'feature':22} {'linear':>10} {'ridge':>10} {'lasso':>10}")
for feature, c_lin, c_ridge, c_lasso in zip(feature_cols, lin_reg.coef_, ridge.coef_, lasso.coef_):
    print(f'{feature:22} {c_lin:10.4f} {c_ridge:10.4f} {c_lasso:10.4f}')

for name, model in [('linear', lin_reg), ('ridge', ridge), ('lasso', lasso)]:
    pred = model.predict(X_val)
    print(f'{name} validation MAE: {mean_absolute_error(y_val, pred):.2f} days')

`total_cost` is built from `total_price` plus `total_freight`, and `total_payment_value` tracks almost the same total (their correlation is 0.9999). These three columns are telling the model nearly the same thing three times, and plain linear regression cannot decide which one deserves the credit. Its coefficients swing wildly to compensate: `total_price` at -1.33, `total_cost` at -1.17, `total_payment_value` at +2.49. These numbers are not wrong, but they are unstable. Rerun the fit on a slightly different sample of orders and they would shift again, even though the model's predictions barely would.

Ridge pulls all three toward each other and toward zero: -0.81, -0.67, and +1.44. Still three active coefficients, but far less extreme. Lasso goes further and sets `total_price`, `total_cost`, `total_payment_value`, and `max_installments` to exactly zero, keeping only `item_count`, `total_freight`, and `estimated_days`. It picked one representative from the overlapping group and dropped the rest.

Notice validation MAE barely moves across all three models, about 5.60 days for linear and ridge, 5.61 for lasso. Regularization did not make this model meaningfully more accurate. What it bought is stability and a simpler, more interpretable set of coefficients, which matters when you need to explain a model's behavior to someone else or trust that it will not swing wildly on the next batch of data.

In [ ]:
# Exercise 4.1
# Refit Lasso with alpha=1.0 instead of 0.1 (ten times the penalty strength).
# Which coefficients are still nonzero? Compare this to the alpha=0.1 result above.

# TODO: your code here


---

## Block 5 — Gradient Descent, How the Fit Actually Happens

`LinearRegression.fit()` finds its coefficients with a closed-form formula, solved in one step. Most other models, including logistic regression and every neural network, cannot be solved that way. They use **gradient descent** instead: start with a guess for the coefficients, measure how wrong the guess is, then nudge the coefficients a small step in the direction that reduces the error. Repeat until the error stops improving.

The size of each step is called the **learning rate**. Too small, and the fit crawls toward the answer, wasting iterations. Too large, and each step overshoots so far that the error grows instead of shrinking, and the fit diverges. To see this directly, we implement gradient descent by hand for a single-feature regression: `estimated_days` predicting `delivery_days`, the same pair of columns Session 3, Block 5 used for the bias-variance demo.

In [ ]:
gd_data = delivered.dropna(subset=['estimated_days', 'delivery_days'])
x_raw = gd_data['estimated_days'].to_numpy(dtype=float)
y = gd_data['delivery_days'].to_numpy(dtype=float)

# Standardize x so the learning rate behaves the same regardless of the feature's
# original units. This is the same reason Blocks 3 and 4 standardized features.
x = (x_raw - x_raw.mean()) / x_raw.std()

# The closed-form answer, for comparison
closed_form = LinearRegression().fit(x.reshape(-1, 1), y)
print(f'Closed-form solution: slope={closed_form.coef_[0]:.4f}, intercept={closed_form.intercept_:.4f}')

In [ ]:
def gradient_descent(x, y, learning_rate, n_iter):
    """Batch gradient descent for y = w*x + b, minimizing mean squared error."""
    n = len(x)
    w, b = 0.0, 0.0
    losses = []
    for _ in range(n_iter):
        pred = w * x + b
        error = pred - y
        losses.append((error ** 2).mean())
        dw = (2 / n) * (error * x).sum()
        db = (2 / n) * error.sum()
        w -= learning_rate * dw
        b -= learning_rate * db
    return w, b, losses

w_slow,  b_slow,  losses_slow  = gradient_descent(x, y, learning_rate=0.01, n_iter=60)
w_good,  b_good,  losses_good  = gradient_descent(x, y, learning_rate=0.1,  n_iter=60)
w_blow,  b_blow,  losses_blow  = gradient_descent(x, y, learning_rate=1.2,  n_iter=6)

print(f'lr=0.01, 60 steps: w={w_slow:.4f}, b={b_slow:.4f}, final loss={losses_slow[-1]:.2f}')
print(f'lr=0.1,  60 steps: w={w_good:.4f}, b={b_good:.4f}, final loss={losses_good[-1]:.2f}')
print(f'lr=1.2,   6 steps: w={w_blow:.4g}, b={b_blow:.4g}, final loss={losses_blow[-1]:.4g}')

All three runs start from the same loss, about 237.5, since they all start from `w=0, b=0`.

At `learning_rate=0.01`, after 60 steps the loss has only dropped to about 92.6, and `w` has reached 2.57, still short of the closed-form answer near 3.66. The steps are real progress, just too small to arrive in time.

At `learning_rate=0.1`, after the same 60 steps, `w` and `b` land almost exactly on the closed-form solution, and the loss reaches about 77.9, matching what a direct formula gives you. This is gradient descent converging: not an approximation, the same answer, reached step by step.

At `learning_rate=1.2`, something different happens. After just 6 steps the loss has exploded to over 4,600, far above where it started, and `w` has swung to roughly -24. Each step overshot the minimum by more than the last one, so the fit is diverging instead of converging. This is why picking a learning rate is a real decision, not a formality: too small wastes time, too large breaks the fit entirely.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_slow, label='learning_rate=0.01 (too slow)')
ax.plot(losses_good, label='learning_rate=0.1 (converges)')
ax.set_xlabel('iteration')
ax.set_ylabel('mean squared error')
ax.set_title('Gradient descent: loss per iteration')
ax.legend()
plt.show()
# learning_rate=1.2 is left off this plot on purpose: its loss reaches the
# thousands within 6 steps, which would flatten the other two curves to the
# bottom of the chart. The printed numbers above already show the divergence.

In [ ]:
# Exercise 5.1
# Try learning_rate=0.05 with n_iter=60. Does it converge to the same w and b as
# learning_rate=0.1? Is it closer to the "too slow" run or the "converges" run?

# TODO: your code here


---

## Challenge Block — Logistic Regression on a New Target: bad_review (Self-Directed Stretch)

> **This block is yours to attempt first.** instructor walks through it live after about 5 to 7 minutes.

### Context

Session 3's Challenge Block introduced `olist_order_reviews_dataset.csv` to demonstrate a leakage trap. Today, that same table comes back for a different job: applying Block 3's logistic regression skill to a target you have not modeled before.

Define `bad_review` as `review_score <= 3`. Unlike `is_late`, this target is not about the order itself, it is about how the customer felt afterward. That means features that would be leaky for predicting `is_late` (like `is_late` itself, or `delivery_days`) are fair game here, since a review is written after delivery, not before.

### Your task (attempt before the walkthrough)

1. Load `olist_order_reviews_dataset.csv`, build `bad_review`, and merge it onto `delivered` using `order_id`.
2. Build a feature set from `is_late`, `delivery_days`, `total_freight`, and `item_count`. Standardize the features.
3. Split into train and validation sets, stratified on `bad_review`, and fit a logistic regression.
4. Report validation accuracy against the majority class baseline, and look at the coefficient signs. Do they make sense given what a late or slow delivery does to a customer's experience?

---

In [ ]:
# Challenge — Step 1: load olist_order_reviews_dataset.csv, build bad_review,
# and merge it onto delivered using order_id

# TODO: your code here


In [ ]:
# Challenge — Step 2: build the feature set (is_late, delivery_days, total_freight,
# item_count) and standardize it with StandardScaler

# TODO: your code here


In [ ]:
# Challenge — Step 3: stratified train/validation split, then fit LogisticRegression

# TODO: your code here


In [ ]:
# Challenge — Step 4: validation accuracy vs. the majority baseline, and your read
# of the coefficient signs

# TODO: your code here


In [ ]:
# ── WALKTHROUGH (Instructor reveals this live) ──
# Step 1: load, build the target, and merge
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
reviews['bad_review'] = reviews['review_score'] <= 3

reviewed = delivered.merge(reviews[['order_id', 'bad_review']], on='order_id', how='inner')
print(f'Orders with a matching review: {len(reviewed):,} of {len(delivered):,}')
print(f'bad_review rate: {reviewed["bad_review"].mean()*100:.2f}%')

In [ ]:
# Steps 2 and 3: feature set, standardize, split, fit
feature_cols = ['is_late', 'delivery_days', 'total_freight', 'item_count']
target_col = 'bad_review'

model_data = reviewed.dropna(subset=feature_cols + [target_col]).copy()
model_data['is_late'] = model_data['is_late'].astype(int)

train, val = train_test_split(
    model_data, test_size=0.20, stratify=model_data[target_col], random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_val = scaler.transform(val[feature_cols])

log_reg = LogisticRegression().fit(X_train, train[target_col])
val_pred = log_reg.predict(X_val)

val_acc = accuracy_score(val[target_col], val_pred)
majority_baseline = 1 - val[target_col].mean()
print(f'Validation accuracy: {val_acc*100:.2f}%')
print(f'Majority baseline:   {majority_baseline*100:.2f}%')
print('Coefficients:')
for feature, coef in zip(feature_cols, log_reg.coef_[0]):
    print(f'  {feature:15}: {coef:8.4f}')

### Step 4: the answer

This model beats its baseline: 81.64% accuracy against a 78.92% majority baseline. Unlike Block 3's `is_late` model, this one has real separating power, because `bad_review` at 21.1% positive is far less imbalanced than `is_late` at 8.1%.

Every coefficient points the direction you would expect. `delivery_days` is the strongest, at +0.45: a longer delivery time raises the predicted probability of a bad review. `is_late` adds another +0.38 on top of that, so being late hurts a review beyond what the raw delivery time already explains. `item_count` is positive too (+0.28), which is a softer signal, perhaps larger orders carry more chances for something to go wrong. `total_freight` is close to zero and slightly negative (-0.03), suggesting shipping cost by itself does not shape how a customer rates the order.

The pattern to carry forward: today's tools, a scaled feature set and a logistic regression, transfer directly to any binary target once you check that the features you are using are actually knowable at the time the prediction needs to happen.

---

## Wrap — What We Built Today

| Skill | Where we used it |
|-------|-------------------|
| Linear regression, fit and coefficient interpretation | Block 2 |
| Logistic regression for a binary target | Block 3 |
| L2 (Ridge) and L1 (Lasso) regularization, coefficient shrinkage vs. feature selection | Block 4 |
| Gradient descent by hand, and how learning rate controls convergence | Block 5 |
| Stretch: logistic regression applied to a new target | Challenge Block |

**The takeaway:** every model today is doing the same basic thing, combining features with learned weights to produce a prediction. Linear regression predicts a number directly. Logistic regression turns that same combination into a probability. Regularization changes how those weights get learned, trading a little training fit for a model that generalizes and explains itself better. Gradient descent is how the weights actually get found when there is no shortcut formula, which is every model from here forward.

---

**Before next class:** finish the exercises you skipped and work through the supplement notebook. Session 5 moves from linear models to trees. You will see the same bias-variance tradeoff from Session 3, and the same overfitting risk that motivated regularization today, show up again in a decision tree that grows too deep.